In [ ]:
pip install -U ckiptagger[tf,gdown]

In [ ]:
pip install keybert

In [ ]:
pip install zhon

In [ ]:
from ckiptagger import data_utils, construct_dictionary, WS, POS, NER
from keybert import KeyBERT
from sklearn.feature_extraction.text import CountVectorizer
from zhon.hanzi import punctuation
import csv

In [ ]:
data_utils.download_data_url("./") # iis-ckip
ws = WS("./data")

In [ ]:
def ws_zh(text):
  words = ws([text])
  texts = words[0]
  for i, text in enumerate(texts):
      texts[i] = texts[i].replace('▲', '')
      for punc in punctuation:
          texts[i] = texts[i].replace(punc, '')

  cleaned_texts = [text for text in texts if text.strip()]
  # print(words[0])

  return cleaned_texts

In [ ]:
vectorizer =CountVectorizer(tokenizer=ws_zh)
kw_model = KeyBERT(model='distiluse-base-multilingual-cased-v1')

In [ ]:
import ast

csv_file = 'cleaned_data.csv'
content_data = []
with open(csv_file, mode='r', newline='') as file:
    reader = csv.DictReader(file)
    for row in reader:
      # print(row['content'])
      text = row['title']+"。"+row['content']
      # print(text)
      content_data.append([text, row['time']])

csv_file = 'keywords.csv'
cnt = 0
for data, time in content_data:
  print(type(time))
  cnt+=1
  docs = data
  keywords = kw_model.extract_keywords(docs,vectorizer=vectorizer, top_n=20, diversity=0.7)

  tuple_string = time

  tuple_data = ast.literal_eval(tuple_string)

  with open(csv_file, mode='a', newline='') as file:
      writer = csv.writer(file)

      if cnt == 1 and file.tell() == 0:
          writer.writerow(['ketwords', 'weight', 'event_date'])
      total = 0.0
      for keyword, weight in keywords:
          total += weight
      for keyword, weight in keywords:
          writer.writerow([keyword, weight/total, tuple_data])
      print(cnt)
      print(f'Saved {csv_file}')


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import datetime


def time_decay_weight(weight, time_diff):
    half_life = datetime.timedelta(days=90)
    decay_factor = 0.5 ** (time_diff / half_life.total_seconds())
    return weight * decay_factor

keywords1 = []
keywords2 = []


with open('keywords.csv', newline='') as csvfile:
    reader = csv.DictReader(csvfile)
    for row in reader:
        key = row['keywords']
        weight = float(row['weight'])
        tuple_string = ast.literal_eval(row['event_date'])
        keywords1.append((key, weight, datetime.datetime(tuple_string[0], tuple_string[1], tuple_string[2])))

result = []
with open('random_data.csv', newline='') as csvfile:
    reader = csv.DictReader(csvfile)
    for row in reader:
        date_parts = row['publish_time'].split()[0].split("-")
        tuple_string = (int(date_parts[0]), int(date_parts[1]), int(date_parts[2]))
        content = row['title'] + "。" + row['content']
        docs = content
        keywords2_tmp = kw_model.extract_keywords(docs,vectorizer=vectorizer, top_n=20, diversity=0.7)

        total = 0
        for word, w in keywords2_tmp:
          total += w
        keywords2 = [(word, w/total, datetime.datetime(tuple_string[0], tuple_string[1], tuple_string[2])) for word, w in keywords2_tmp]
        print(keywords1)
        print(keywords2)

        current_time = datetime.datetime(2023, 9, 1)
        #current_time = datetime.datetime.now()

        tfidf_vectorizer = TfidfVectorizer(lowercase=False)

        keywords1_str = ""
        keywords2_str = ""
        weights1 = []
        weights2 = []

        for keyword, weight, time in keywords1:
            time_diff = (current_time - time).total_seconds()
            decayed_weight = time_decay_weight(weight, time_diff)
            keywords1_str += keyword + " "
            weights1.append(decayed_weight)

        for keyword, weight, time in keywords2:
            time_diff = (current_time - time).total_seconds()
            decayed_weight = time_decay_weight(weight, time_diff)
            keywords2_str += keyword + " "
            weights2.append(decayed_weight)

        tfidf_matrix = tfidf_vectorizer.fit_transform([keywords1_str, keywords2_str])

        for i, keyword in enumerate(keywords1):
            if keyword[0] in tfidf_vectorizer.vocabulary_:
                tfidf_matrix[0, tfidf_vectorizer.vocabulary_[keyword[0]]] *= weights1[i]

        for i, keyword in enumerate(keywords2):
            if keyword[0] in tfidf_vectorizer.vocabulary_:
                tfidf_matrix[1, tfidf_vectorizer.vocabulary_[keyword[0]]] *= weights2[i]

        cosine_sim = cosine_similarity(tfidf_matrix[0], tfidf_matrix[1])

        result.append([cosine_sim[0][0], row['type'], row['title']])
        # print("cosine similarity：", cosine_sim[0][0])

#print the sorted cosine similarity
arrange_result = sorted(result, key=lambda x: x[0])
for score, types, title in arrange_result:
  print(score, " ", types, " ", title)
